# Causal Inference — Exercises

Companion to the note [Causal Inference](Causal%20Inference.md).

Practise potential outcomes and the ATE, do-calculus intuition and backdoor adjustment, Simpson's paradox, collider bias, propensity scores and IPW, quasi-experiments (diff-in-diff, instrumental variables) and meta-learners for heterogeneous effects: by hand and with simulations where the true effect is known.

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.linear_model import LinearRegression, LogisticRegression
from scipy.stats import ttest_ind
rng = np.random.default_rng(11)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Part A · Concepts

### Exercise 1 · Seeing vs. doing
*🧠 Concept · ★☆☆*

Ice-cream sales and drownings are strongly correlated. Explain the difference between $p(\text{drown}\mid \text{ice cream}=\text{high})$
and $p(\text{drown}\mid do(\text{ice cream}=\text{high}))$, draw the DAG you have in mind, and say which of the two quantities a
recommender team needs when it asks "would this user have clicked *without* our recommendation?".

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What common cause drives both variables? What does the do-operator do to the arrows into the intervened node?

</details>

<details>
<summary><b>✅ Solution</b></summary>

DAG: $\text{temperature}\to\text{ice cream}$, $\text{temperature}\to\text{drownings}$. Conditioning on high sales tells us it is probably hot,
so drownings are likely: the observational conditional is high. Intervening, $do(\text{ice cream}=\text{high})$, cuts the arrow
$\text{temperature}\to\text{ice cream}$; temperature keeps its natural distribution and drownings do not change, so the interventional quantity equals $p(\text{drown})$.
The recommender question is interventional (a counterfactual comparison of $Y(1)$ vs. $Y(0)$), so it needs $do(\cdot)$, not a click-prediction model.

</details>

### Exercise 2 · The fundamental problem
*🧠 Concept · ★☆☆*

In the potential-outcomes framework each unit has $Y_i(1)$ and $Y_i(0)$ but we observe only $Y_i = T_iY_i(1)+(1-T_i)Y_i(0)$.

1. Why can the individual effect $Y_i(1)-Y_i(0)$ never be computed from data?
2. Which assumptions make $\mathrm{ATE}=\mathbb E[Y(1)-Y(0)]$ identifiable from observational data? Name and explain **consistency/SUTVA**, **ignorability** and **positivity**.
3. Why does randomisation give ignorability for free?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ignorability: $(Y(1),Y(0))\perp T\mid X$. Positivity: $0<P(T=1\mid X)<1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Each unit is either treated or not; the other potential outcome is a *counterfactual* that is missing by construction.
2. **Consistency/SUTVA**: the observed outcome equals the potential outcome of the treatment actually received, and one unit's treatment does not affect another's outcome (no interference, one version of treatment).
   **Ignorability** (no unmeasured confounding): given $X$, treatment assignment is independent of the potential outcomes.
   **Positivity**: every $X$ has a non-zero chance of both treatments, otherwise we have nobody to compare with.
   Then $\mathrm{ATE}=\mathbb E_X\big[\mathbb E[Y\mid T=1,X]-\mathbb E[Y\mid T=0,X]\big]$.
3. A coin flip is independent of everything, including $(Y(1),Y(0))$, so $\mathbb E[Y\mid T=1]-\mathbb E[Y\mid T=0]$ is already unbiased.

</details>

### Exercise 3 · Collider bias
*🧠 Concept · ★★☆*

Among Hollywood actors, talent and good looks appear *negatively* correlated, although in the general population they are independent.
Explain this with a DAG and d-separation. Then give a machine-learning example of the same bias (hint: datasets that only contain
items that were *shown* or *clicked*).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Being cast depends on talent **and** looks. What happens to a v-structure $A\to C\leftarrow B$ when you condition on $C$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

DAG: $\text{talent}\to\text{cast}\leftarrow\text{looks}$. "Cast" is a **collider**: the path talent–looks is blocked marginally, but
conditioning on the collider (studying only actors) **opens** it. Among those cast, a less talented actor must be better-looking to have
made it ("explaining away", as in [[Probabilistic Graphical Models]]).
ML example: a recommender log only contains items that were exposed because they were popular *or* very relevant; within the log,
popularity and relevance look negatively related. This **selection bias** is why IPW-corrected evaluation is used in [[Evaluating Recommenders]].
You will simulate it in Exercise 16.

</details>

### Exercise 4 · Choosing a backdoor adjustment set
*🧠 Concept · ★★☆*

DAG: $C\to X$, $C\to Y$, $X\to M\to Y$, $X\to Y$, $Y\to S$. We want the total effect of $X$ on $Y$.
Which of the sets $\varnothing$, $\{C\}$, $\{C,M\}$, $\{C,S\}$ satisfy the backdoor criterion? Explain each answer.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The backdoor criterion: (i) no node in the set is a descendant of $X$; (ii) the set blocks every path from $X$ to $Y$ that starts with an arrow **into** $X$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- $\varnothing$: **no**, the backdoor path $X\leftarrow C\to Y$ is open (confounding).
- $\{C\}$: **yes**, blocks the only backdoor path and contains no descendant of $X$; then $p(y\mid do(x))=\sum_c p(y\mid x,c)p(c)$.
- $\{C,M\}$: **no**, $M$ is a descendant (mediator) of $X$; adjusting for it removes the indirect effect $X\to M\to Y$ and estimates only the direct effect.
- $\{C,S\}$: **no**, $S$ is a descendant of $X$ (via $Y$); conditioning on a descendant of the outcome induces bias.

</details>

### Exercise 5 · Which quasi-experiment?
*🧠 Concept · ★★☆*

Pick the most suitable design from the note (instrumental variable, difference-in-differences, regression discontinuity) and state its key assumption:

(a) A scholarship is given to every student with an entrance score $\ge 80$; does it raise graduation rates?
(b) One city introduces a sugar tax in 2024, a similar city does not; we have consumption for both cities in 2023 and 2025.
(c) Users were randomly *offered* a premium trial, but only some accepted; what is the effect of premium on retention?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look for a cutoff, a before/after × treated/control structure, or a random nudge that only shifts the treatment.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Regression discontinuity**: compare students just above and just below 80. Assumption: units cannot precisely manipulate the score around the cutoff, so they are comparable on both sides (continuity of potential outcomes).
(b) **Difference-in-differences**: (change in treated city) − (change in control city). Assumption: **parallel trends**: without the tax, both cities' consumption would have changed by the same amount.
(c) **Instrumental variable**: the random offer $Z$ shifts premium uptake $X$. Assumptions: relevance ($Z$ affects $X$), exclusion (the offer affects retention only through premium), independence of $Z$; with one-sided compliance this estimates the effect on compliers (LATE).

</details>

## Part B · By hand

### Exercise 6 · ATE vs. naive difference
*✍️ By hand · ★☆☆*

An oracle shows both potential outcomes for six users (outcome = minutes watched per day):

| unit | $Y(1)$ | $Y(0)$ | $T$ |
|---|---|---|---|
| 1 | 7 | 5 | 1 |
| 2 | 6 | 4 | 1 |
| 3 | 8 | 6 | 1 |
| 4 | 3 | 2 | 0 |
| 5 | 2 | 1 | 0 |
| 6 | 4 | 2 | 0 |

Compute (a) the true ATE, (b) the naive difference of observed means $\mathbb E[Y\mid T=1]-\mathbb E[Y\mid T=0]$ and
(c) decompose (b) as ATT + selection bias, where selection bias $=\mathbb E[Y(0)\mid T=1]-\mathbb E[Y(0)\mid T=0]$.

In [ ]:
ate_true = None
naive_diff = None
selection_bias = None

_Y1 = np.array([7, 6, 8, 3, 2, 4.]); _Y0 = np.array([5, 4, 6, 2, 1, 2.]); _T = np.array([1, 1, 1, 0, 0, 0])
check('ATE', ate_true, np.mean(_Y1 - _Y0))
check('naive difference', naive_diff, _Y1[_T == 1].mean() - _Y0[_T == 0].mean())
check('selection bias', selection_bias, _Y0[_T == 1].mean() - _Y0[_T == 0].mean())

<details>
<summary><b>💡 Hint</b></summary>

Observed outcomes: units 1–3 show $Y(1)$, units 4–6 show $Y(0)$. ATT is the mean effect among treated units 1–3.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Effects $2,2,2,1,1,2$, so $\mathrm{ATE}=10/6\approx 1.667$.
(b) $\mathbb E[Y\mid T=1]=(7+6+8)/3=7$, $\mathbb E[Y\mid T=0]=(2+1+2)/3=1.667$, difference $5.333$.
(c) $\mathrm{ATT}=2$, selection bias $=\frac{5+4+6}{3}-\frac{2+1+2}{3}=5-1.667=3.333$, and $2+3.333=5.333$ ✔.
Heavy users were the ones who got treated, so the naive comparison more than triples the true effect.

```python
ate_true = 10/6
naive_diff = 7 - 5/3
selection_bias = 5 - 5/3
```

</details>

### Exercise 7 · Simpson's paradox (kidney stones)
*✍️ By hand · ★★☆*

Success counts of two treatments, split by stone size $Z$:

| | small stones | large stones | total |
|---|---|---|---|
| A | 81/87 | 192/263 | 273/350 |
| B | 234/270 | 55/80 | 289/350 |

1. Compute the naive difference in success rates $P(\text{succ}\mid A)-P(\text{succ}\mid B)$.
2. Assume stone size is the only confounder ($Z\to$ treatment, $Z\to$ outcome). Use the backdoor formula
$P(\text{succ}\mid do(t))=\sum_z P(\text{succ}\mid t,z)P(z)$ with $P(z)$ from all 700 patients, and compute the adjusted difference.

In [ ]:
naive_AB = None
adjusted_AB = None

_s = {'A': (81, 87, 192, 263), 'B': (234, 270, 55, 80)}
_pz = np.array([87 + 270, 263 + 80]) / 700
_do = {k: _pz @ np.array([v[0] / v[1], v[2] / v[3]]) for k, v in _s.items()}
check('naive A-B', naive_AB, 273/350 - 289/350, tol=1e-3)
check('adjusted A-B', adjusted_AB, _do['A'] - _do['B'], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$P(\text{small}) = (87+270)/700 = 0.51$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$P(\text{succ}\mid do(A)) = 0.51\cdot\frac{81}{87}+0.49\cdot\frac{192}{263}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $0.780 - 0.826 = -0.0457$: B looks better.
2. $P(\text{succ}\mid do(A)) = 0.51\cdot0.931+0.49\cdot0.730=0.8325$, $P(\text{succ}\mid do(B)) = 0.51\cdot0.867+0.49\cdot0.6875=0.7789$,
adjusted difference $=+0.0537$: A is better in **each** stratum and after adjustment.
The reversal happens because A was given mostly to the hard (large-stone) cases. Whether to adjust depends on the DAG, not on the numbers.

```python
naive_AB = 273/350 - 289/350
adjusted_AB = (0.51*81/87 + 0.49*192/263) - (0.51*234/270 + 0.49*55/80)
```

</details>

### Exercise 8 · IPW by hand
*✍️ By hand · ★★☆*

Six observations $(X, T, Y)$: $(1,1,10)$, $(1,1,12)$, $(1,0,8)$, $(0,1,6)$, $(0,0,3)$, $(0,0,5)$.

1. Estimate the propensity $e(x)=P(T=1\mid X=x)$ by the within-group frequency.
2. Compute the IPW estimate $\widehat{\mathrm{ATE}}=\frac1n\sum_i\Big[\frac{T_iY_i}{e(X_i)}-\frac{(1-T_i)Y_i}{1-e(X_i)}\Big]$.
3. Compare with stratification: average the within-stratum mean differences weighted by $P(X)$.

In [ ]:
ipw_hand = None

_D = np.array([[1, 1, 10], [1, 1, 12], [1, 0, 8], [0, 1, 6], [0, 0, 3], [0, 0, 5.]])
_strat = sum(np.mean(_D[:, 0] == x) * (_D[(_D[:, 0] == x) & (_D[:, 1] == 1), 2].mean() - _D[(_D[:, 0] == x) & (_D[:, 1] == 0), 2].mean()) for x in (0, 1))
check('IPW estimate (equals stratification here)', ipw_hand, _strat)

<details>
<summary><b>💡 Hint</b></summary>

$e(1)=2/3$, $e(0)=1/3$. The treated terms are $10\cdot\frac32$, $12\cdot\frac32$, $6\cdot 3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $e(1)=2/3$, $e(0)=1/3$.
2. Treated: $15+18+18=51$. Control: $8/(1/3)+3/(2/3)+5/(2/3)=24+4.5+7.5=36$. $\widehat{\mathrm{ATE}}=(51-36)/6=2.5$.
3. Stratum $X=1$: $11-8=3$; stratum $X=0$: $6-4=2$; $P(X=1)=P(X=0)=\tfrac12$, so $2.5$. ✔
With propensities estimated by stratum frequencies, IPW and stratification coincide; with a parametric propensity model they differ.

```python
ipw_hand = 2.5
```

</details>

### Exercise 9 · Difference-in-differences
*✍️ By hand · ★☆☆*

Average weekly sessions per user: the treated region goes from 10 (before) to 15 (after a new feature), the control region from 8 to 10.
Compute the DiD estimate and say what the naive before/after comparison in the treated region would have claimed.

In [ ]:
did = None
naive_before_after = None

check('DiD', did, (15 - 10) - (10 - 8))
check('naive before/after', naive_before_after, 15 - 10)

<details>
<summary><b>💡 Hint</b></summary>

DiD = (treated after − treated before) − (control after − control before).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{DiD}=(15-10)-(10-8)=3$. The naive before/after estimate $5$ wrongly attributes the common trend $(+2)$, e.g. seasonality, to the feature.
DiD is only valid under **parallel trends**; check pre-period trends if you have several periods.

```python
did = 3
naive_before_after = 5
```

</details>

### Exercise 10 · The Wald (IV) estimator
*✍️ By hand · ★★☆*

A random encouragement $Z\in\{0,1\}$ raises the share taking a treatment $X$ from $0.2$ ($Z=0$) to $0.7$ ($Z=1$).
Mean outcomes are $\mathbb E[Y\mid Z=0]=1.0$ and $\mathbb E[Y\mid Z=1]=2.0$.
Compute the IV (Wald) estimate of the effect of $X$ on $Y$ and derive the formula from $Y=\beta X+U$ with $Z\perp U$.

In [ ]:
wald = None

check('Wald estimate', wald, (2.0 - 1.0) / (0.7 - 0.2))

<details>
<summary><b>💡 Hint</b></summary>

Take $\mathbb E[\cdot\mid Z=1]-\mathbb E[\cdot\mid Z=0]$ of both sides of $Y=\beta X+U$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[Y\mid Z=1]-\mathbb E[Y\mid Z=0] = \beta\,(\mathbb E[X\mid Z=1]-\mathbb E[X\mid Z=0]) + 0$ since $\mathbb E[U\mid Z]$ is constant.
So $\beta=\frac{2.0-1.0}{0.7-0.2}=2$. The "intent-to-treat" effect $1.0$ is scaled up by the compliance rate $0.5$.
A weak instrument (denominator near 0) makes the estimate explode, which is the IV analogue of dividing by noise.

```python
wald = 2.0
```

</details>

### Exercise 11 · Why IPW is unbiased
*✍️ By hand · ★★★*

Assume consistency, ignorability $(Y(1),Y(0))\perp T\mid X$, and positivity, with the **true** propensity $e(X)$.
Show that $\mathbb E\Big[\frac{TY}{e(X)}\Big]=\mathbb E[Y(1)]$. What goes wrong numerically when $e(X)$ is close to 0 or 1?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Use $TY=TY(1)$ and the tower rule $\mathbb E[\cdot]=\mathbb E[\mathbb E[\cdot\mid X]]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E\Big[\frac{TY}{e(X)}\Big]=\mathbb E\Big[\frac{TY(1)}{e(X)}\Big]=\mathbb E\Big[\frac{\mathbb E[T\mid X]\,\mathbb E[Y(1)\mid X]}{e(X)}\Big]
=\mathbb E\big[\mathbb E[Y(1)\mid X]\big]=\mathbb E[Y(1)]$,
using consistency, then ignorability to factorise the conditional expectation, then $\mathbb E[T\mid X]=e(X)$. The same argument with $1-T$ gives $\mathbb E[Y(0)]$.
Near-violations of positivity give huge weights $1/e(X)$, so a handful of units dominate and the variance explodes. Remedies: clipping/trimming
propensities, stabilised or self-normalised weights, or doubly robust estimators.

</details>

## Part C · Simulations where the truth is known

### Exercise 12 · Analysing an A/B test
*💻 Code · ★☆☆*

Simulate an A/B test with 2000 users per arm: control outcomes $\sim\mathcal N(10, 3^2)$, treatment $\sim\mathcal N(10.4, 3^2)$.
Compute the difference in means `ab_diff` and a 95% confidence interval `ab_ci = (low, high)` using the normal approximation
$\hat\Delta\pm1.96\sqrt{s_1^2/n_1+s_0^2/n_0}$. Does the interval exclude 0?

In [ ]:
y_ctrl = rng.normal(10.0, 3.0, 2000)
y_trt = rng.normal(10.4, 3.0, 2000)
ab_diff = None
ab_ci = None

_r = ttest_ind(y_trt, y_ctrl, equal_var=False)
check('difference in means', ab_diff, y_trt.mean() - y_ctrl.mean())
check('95% CI', ab_ci, tuple(_r.confidence_interval()), tol=5e-3)

<details>
<summary><b>💡 Hint</b></summary>

Use `ddof=1` for the sample variances.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Because assignment is random, the difference in means is an unbiased ATE estimate with no adjustment needed. The standard error is about
$\sqrt{9/2000+9/2000}\approx0.095$, so a true effect of $0.4$ is detected comfortably (the CI excludes 0).
The normal CI matches scipy's Welch $t$-interval to about 3 decimals at this sample size.

```python
y_ctrl = rng.normal(10.0, 3.0, 2000)
y_trt = rng.normal(10.4, 3.0, 2000)
ab_diff = y_trt.mean() - y_ctrl.mean()
se = np.sqrt(y_trt.var(ddof=1) / len(y_trt) + y_ctrl.var(ddof=1) / len(y_ctrl))
ab_ci = (ab_diff - 1.96 * se, ab_diff + 1.96 * se)
print(ab_diff, ab_ci)
```

</details>

### Exercise 13 · Confounding and regression adjustment
*💻 Code · ★☆☆*

Simulate $n=5000$ units with $Z\sim\mathcal N(0,1)$, $X=Z+\mathcal N(0,1)$, $Y=2X+3Z+\mathcal N(0,1)$. The causal effect of $X$ is 2.
Fit (a) $Y\sim X$ and (b) $Y\sim X+Z$ by least squares and store the coefficient of $X$ as `naive_coef` and `adjusted_coef`.
Predict the naive value analytically before running.

In [ ]:
n = 5000
Z = rng.normal(size=n)
X = Z + rng.normal(size=n)
Y = 2 * X + 3 * Z + rng.normal(size=n)
naive_coef = None
adjusted_coef = None

check('adjusted coefficient ≈ causal effect 2', adjusted_coef, 2.0, tol=0.06)
check('naive coefficient ≈ 2 + 3·Cov(X,Z)/Var(X) = 3.5', naive_coef, 3.5, tol=0.1)

<details>
<summary><b>💡 Hint</b></summary>

Omitted-variable bias: the naive slope is $2+3\,\mathrm{Cov}(X,Z)/\mathrm{Var}(X)$ with $\mathrm{Cov}(X,Z)=1$, $\mathrm{Var}(X)=2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Naive slope $\approx 2+3\cdot\tfrac12=3.5$; including the confounder $Z$ (a valid backdoor set) recovers $\approx2$.
Regression adjustment is backdoor adjustment under a linear model. It fails if $Z$ is unmeasured, which is where IV comes in (Exercise 17).

```python
n = 5000
Z = rng.normal(size=n)
X = Z + rng.normal(size=n)
Y = 2 * X + 3 * Z + rng.normal(size=n)
naive_coef = LinearRegression().fit(X[:, None], Y).coef_[0]
adjusted_coef = LinearRegression().fit(np.c_[X, Z], Y).coef_[0]
print(naive_coef, adjusted_coef)
```

</details>

### Exercise 14 · Propensity scores and IPW
*💻 Code · ★★☆*

Observational data: $X\in\mathbb R^2$ standard normal, $T\sim\mathrm{Bernoulli}(\sigma(1.5x_1-0.5x_2))$,
$Y = x_1 + 2x_2 + T(1.5+x_1) + \varepsilon$. The true ATE is $1.5$ (since $\mathbb E[x_1]=0$).
Implement `ipw_ate(X, T, Y, clip=0.01)` that fits a logistic-regression propensity model, clips $e$ to $[\text{clip},1-\text{clip}]$ and returns the
**self-normalised** (Hájek) IPW estimate. Compare with the naive difference in means.

In [ ]:
n = 20000
Xo = rng.normal(size=(n, 2))
To = rng.binomial(1, sigmoid(1.5 * Xo[:, 0] - 0.5 * Xo[:, 1]))
Yo = Xo[:, 0] + 2 * Xo[:, 1] + To * (1.5 + Xo[:, 0]) + rng.normal(size=n)

def ipw_ate(X, T, Y, clip=0.01):
    # TODO: e = P(T=1|X) from LogisticRegression; weights T/e and (1-T)/(1-e); weighted means
    return None

ate_ipw = ipw_ate(Xo, To, Yo)
ate_naive = None

check('IPW recovers ATE = 1.5', ate_ipw, 1.5, tol=0.12)
if ate_naive is not None:
    check('naive estimate is badly biased', abs(ate_naive - 1.5) > 0.5, True)
else:
    check('naive estimate is badly biased', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Hájek: $\hat\mu_1=\frac{\sum T_iY_i/e_i}{\sum T_i/e_i}$, $\hat\mu_0=\frac{\sum(1-T_i)Y_i/(1-e_i)}{\sum(1-T_i)/(1-e_i)}$, ATE $=\hat\mu_1-\hat\mu_0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Units with large $x_1$ are both more likely to be treated and have higher outcomes **and** larger effects, so the naive difference is far above 1.5.
IPW reweights each group to look like the full population and lands close to 1.5. Self-normalisation makes the estimator more stable than the
plain Horvitz–Thompson version, and clipping guards against near-zero propensities (Exercise 11). Always inspect the propensity histogram for overlap.

```python
n = 20000
Xo = rng.normal(size=(n, 2))
To = rng.binomial(1, sigmoid(1.5 * Xo[:, 0] - 0.5 * Xo[:, 1]))
Yo = Xo[:, 0] + 2 * Xo[:, 1] + To * (1.5 + Xo[:, 0]) + rng.normal(size=n)

def ipw_ate(X, T, Y, clip=0.01):
    e = LogisticRegression().fit(X, T).predict_proba(X)[:, 1]
    e = np.clip(e, clip, 1 - clip)
    w1, w0 = T / e, (1 - T) / (1 - e)
    return np.sum(w1 * Y) / np.sum(w1) - np.sum(w0 * Y) / np.sum(w0)

ate_ipw = ipw_ate(Xo, To, Yo)
ate_naive = Yo[To == 1].mean() - Yo[To == 0].mean()
print(ate_ipw, ate_naive)
```

</details>

### Exercise 15 · T-learner vs. S-learner
*💻 Code · ★★☆*

Heterogeneous effects: $x\sim U[-1,1]$, $T\sim\mathrm{Bernoulli}(0.5)$, $Y = x + T\,\tau(x)+\varepsilon$ with $\tau(x)=1+2x$.

- **T-learner**: fit separate linear regressions $\hat\mu_1$ on treated and $\hat\mu_0$ on control; $\hat\tau(x)=\hat\mu_1(x)-\hat\mu_0(x)$.
- **S-learner**: fit one linear regression on features $[x, T]$; $\hat\tau(x)=\hat\mu(x,1)-\hat\mu(x,0)$.

Evaluate both on `x_grid` and store `tau_T` and `tau_S`. Why does the S-learner fail here, and how would you fix it?

In [ ]:
n = 4000
xh = rng.uniform(-1, 1, n)
Th = rng.binomial(1, 0.5, n)
Yh = xh + Th * (1 + 2 * xh) + 0.5 * rng.normal(size=n)
x_grid = np.linspace(-1, 1, 5)
tau_T = None
tau_S = None

check('T-learner recovers tau(x) = 1 + 2x', tau_T, 1 + 2 * x_grid, tol=0.15)
if tau_S is not None:
    check('linear S-learner gives a constant effect', np.ptp(tau_S), 0.0, tol=1e-8)
else:
    check('linear S-learner gives a constant effect', None, 0.0)

<details>
<summary><b>💡 Hint</b></summary>

For the linear S-learner, $\hat\mu(x,1)-\hat\mu(x,0)$ is just the coefficient of $T$, whatever $x$ is.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The T-learner recovers $\tau(x)\approx1+2x$. A **linear** S-learner has no $x\cdot T$ interaction, so its CATE is the constant coefficient of $T$
(≈ the ATE $=1$). Fix: add the interaction feature $xT$, or use a flexible model (trees, boosting) as the S-learner's base learner.
The trade-off: S-learners share data across arms (good when effects are small), T-learners can overfit differences when one arm is small;
the X-learner and causal forests address both.

```python
n = 4000
xh = rng.uniform(-1, 1, n)
Th = rng.binomial(1, 0.5, n)
Yh = xh + Th * (1 + 2 * xh) + 0.5 * rng.normal(size=n)
x_grid = np.linspace(-1, 1, 5)
m1 = LinearRegression().fit(xh[Th == 1, None], Yh[Th == 1])
m0 = LinearRegression().fit(xh[Th == 0, None], Yh[Th == 0])
tau_T = m1.predict(x_grid[:, None]) - m0.predict(x_grid[:, None])
s = LinearRegression().fit(np.c_[xh, Th], Yh)
tau_S = s.predict(np.c_[x_grid, np.ones(5)]) - s.predict(np.c_[x_grid, np.zeros(5)])
print(np.round(tau_T, 2), np.round(tau_S, 2))
```

</details>

### Exercise 16 · Simulating collider bias
*💻 Code · ★★☆*

Draw independent "talent" $A$ and "looks" $B$, both $\mathcal N(0,1)$, $n=20000$. An actor is cast if $A+B+\mathcal N(0,0.5^2)>1.5$.
Store the correlation of $A$ and $B$ in the full population (`corr_all`) and among the cast (`corr_cast`).

In [ ]:
n = 20000
A = rng.normal(size=n)
B = rng.normal(size=n)
corr_all = None
corr_cast = None

check('no correlation in the population', corr_all, 0.0, tol=0.03)
if corr_cast is not None:
    check('strong negative correlation after selection', corr_cast < -0.4, True)
else:
    check('strong negative correlation after selection', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`cast = A + B + 0.5 * rng.normal(size=n) > 1.5`, then `np.corrcoef(A[cast], B[cast])[0, 1]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In the population the correlation is $\approx0$; among the cast it is strongly negative ($\approx-0.6$). Nothing causal links $A$ and $B$:
selecting on their common effect alone creates the association. The same thing happens when you train on data filtered by a past model's decisions.

```python
n = 20000
A = rng.normal(size=n)
B = rng.normal(size=n)
cast = A + B + 0.5 * rng.normal(size=n) > 1.5
corr_all = np.corrcoef(A, B)[0, 1]
corr_cast = np.corrcoef(A[cast], B[cast])[0, 1]
print(corr_all, corr_cast)
```

</details>

### Exercise 17 · Two-stage least squares
*💻 Code · ★★★*

Unobserved confounder $U$: $Z\sim\mathcal N(0,1)$ (instrument), $U\sim\mathcal N(0,1)$,
$X = 0.8Z + U + \mathcal N(0,1)$, $Y = 1.5X - 2U + \mathcal N(0,1)$. The causal effect is 1.5.
Implement `tsls(Z, X, Y)`: (1) regress $X$ on $Z$ (with intercept), (2) regress $Y$ on the fitted $\hat X$; return the slope.
Store the OLS slope of $Y$ on $X$ in `ols_slope` and the 2SLS estimate in `iv_slope`.

In [ ]:
n = 20000
Zi = rng.normal(size=n); U = rng.normal(size=n)
Xi = 0.8 * Zi + U + rng.normal(size=n)
Yi = 1.5 * Xi - 2 * U + rng.normal(size=n)

def tsls(Z, X, Y):
    # TODO
    return None

iv_slope = tsls(Zi, Xi, Yi)
ols_slope = None

check('2SLS ≈ 1.5', iv_slope, 1.5, tol=0.08)
check('2SLS equals Cov(Z,Y)/Cov(Z,X)', iv_slope, np.cov(Zi, Yi)[0, 1] / np.cov(Zi, Xi)[0, 1], tol=1e-6)
check('OLS is biased: 1.5 - 2·Var(U)/Var(X)', ols_slope, 1.5 - 2 / (0.64 + 1 + 1), tol=0.05)

<details>
<summary><b>💡 Hint 1</b></summary>

Stage 1: `Xhat = LinearRegression().fit(Z[:, None], X).predict(Z[:, None])`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

With one instrument, 2SLS reduces to $\mathrm{Cov}(Z,Y)/\mathrm{Cov}(Z,X)$, the continuous version of the Wald estimator.

</details>

<details>
<summary><b>✅ Solution</b></summary>

OLS gives $1.5+\mathrm{Cov}(X,-2U)/\mathrm{Var}(X)=1.5-2/2.64\approx0.74$, about half the truth, because $U$ raises $X$ and lowers $Y$.
2SLS uses only the variation in $X$ driven by $Z$, which is independent of $U$, and recovers $\approx1.5$.
Note: the standard errors from the naive second-stage regression are wrong; use proper IV standard errors in practice.

```python
n = 20000
Zi = rng.normal(size=n); U = rng.normal(size=n)
Xi = 0.8 * Zi + U + rng.normal(size=n)
Yi = 1.5 * Xi - 2 * U + rng.normal(size=n)

def tsls(Z, X, Y):
    Xhat = LinearRegression().fit(Z[:, None], X).predict(Z[:, None])
    return LinearRegression().fit(Xhat[:, None], Y).coef_[0]

iv_slope = tsls(Zi, Xi, Yi)
ols_slope = LinearRegression().fit(Xi[:, None], Yi).coef_[0]
print(ols_slope, iv_slope)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Causal Inference](Causal%20Inference.md).*